# R2-09 · Despliegue, ficha de modelo y monitoreo — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R2 · *Científico de Datos* · Semana 15 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Convertir un modelo guardado en una **función de predicción** con **validación de entrada**.
2. **Puntuar por lotes** datos nuevos con **faltantes reales** sin inventar predicciones.
3. **Monitorear** el desempeño en producción y el ***drift*** de los datos de entrada.
4. Documentar el modelo en una **ficha de modelo** honesta.

**Competencia de salida:** dejar un modelo usable y vigilado: predicción en línea y por lotes, entregable, monitoreo y ficha.

**Dato real:** PM2.5 diario de 14 estaciones SINCA (2020–2025) y sus datos nuevos de 2026 como «producción».

**Entregable:** que las **7 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import numpy as np
import pandas as pd

# "En vivo o caché": usa los archivos locales; si faltan (ej. Colab), se bajan del repo.
for archivo in ["aire_diario.csv", "aire_nuevos.csv"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{archivo}", archivo)

historia = pd.read_csv("aire_diario.csv", parse_dates=["fecha"])  # 2020–2025: lo que el modelo conoce
nuevos = pd.read_csv("aire_nuevos.csv", parse_dates=["fecha"])    # 2026: "producción", nunca lo vio
print("historia:", historia.shape, historia["fecha"].min().date(), "→", historia["fecha"].max().date())
print("nuevos:  ", nuevos.shape, nuevos["fecha"].min().date(), "→", nuevos["fecha"].max().date())

## Un modelo en un cuaderno no le sirve a nadie

**Desplegar** es poner el modelo a trabajar para otras personas o sistemas, sobre datos que
**todavía no existían** cuando lo entrenaste. En este módulo harás el recorrido completo con un
caso real: un **pronóstico de PM2.5 del día siguiente** para 14 estaciones de monitoreo del SINCA.

Hay dos formas de usar un modelo desplegado:

- **En línea (uno a uno):** llega **un** caso y se necesita la respuesta al instante — por ejemplo,
  un formulario donde alguien de la SEREMI escribe el PM2.5 de hoy y ve el pronóstico de mañana.
- **Por lotes (*batch*):** llega una **tabla** con muchos casos — por ejemplo, todas las estaciones
  y todos los días de 2026 — y se devuelve un archivo con las predicciones.

Y hay una tercera parte que casi siempre se olvida: **cuidar el modelo** cuando ya está andando
(monitoreo, *drift*, faltantes) y **documentarlo** con una ficha honesta.

> **Producción de verdad:** `aire_nuevos.csv` trae enero–septiembre de **2026**, con los huecos
> reales de las estaciones (El Bosque perdió casi la mitad de sus días). El modelo nunca lo vio.

## 0. El modelo que vas a desplegar

En R2-08 encadenaste pasos en un **pipeline** de PM2.5 y lo guardaste con `joblib`. Aquí
reconstruimos, en una sola celda y sin depender de aquel archivo, una **variante más simple**
pensada para producción: **sin meteorología** (solo 7 de las 14 estaciones la miden, y un
imputador la «rellenaría» en silencio en las otras 7). La celda ya está hecha: ejecútala y lee
los comentarios.

- **Objetivo:** el PM2.5 de **mañana** (`pm25_manana`, µg/m³) en cada estación.
- **Features:** la estación, el PM2.5 de hoy y de ayer, la media de los últimos 7 días y el mes.
- **Partición temporal:** entrena con 2020–2024 y se prueba con 2025 (nunca al revés).
- **La receta de features es UNA sola función** (`construir_features`), que se usa igual al
  entrenar y en producción. Si las dos recetas difieren, el modelo falla **sin avisar**.

In [ ]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

FEATURES = ["estacion", "pm25_hoy", "pm25_ayer", "media_7d", "mes"]
OBJETIVO = "pm25_manana"

def construir_features(df):
    """La MISMA receta para entrenar y para producción."""
    partes = []
    for estacion, d in df.groupby("estacion"):
        d = d.set_index("fecha").sort_index().asfreq("D")   # calendario completo: un día sin dato queda NaN
        d["estacion"] = estacion
        d["pm25_hoy"] = d["pm25"]
        d["pm25_ayer"] = d["pm25"].shift(1)
        d["media_7d"] = d["pm25"].rolling(7, min_periods=5).mean()
        d["mes"] = d.index.month
        d["pm25_manana"] = d["pm25"].shift(-1)               # lo que queremos predecir
        partes.append(d.reset_index())
    return pd.concat(partes, ignore_index=True)

# Historia + nuevos juntos: en producción el modelo SÍ conoce los días anteriores a hoy.
todo = construir_features(pd.concat([historia, nuevos], ignore_index=True))
completo = todo[FEATURES + [OBJETIVO]].notna().all(axis=1)
entrenamiento = todo[(todo["fecha"] < "2025-01-01") & completo]
prueba = todo[(todo["fecha"].dt.year == 2025) & completo]
produccion = todo[todo["fecha"] >= "2026-01-01"].reset_index(drop=True)

pre = ColumnTransformer([("est", OneHotEncoder(handle_unknown="ignore"), ["estacion"])],
                        remainder="passthrough")
pipeline = Pipeline([("pre", pre),
                     ("bosque", RandomForestRegressor(n_estimators=100, min_samples_leaf=10,
                                                      random_state=42, n_jobs=-1))])
pipeline.fit(entrenamiento[FEATURES], entrenamiento[OBJETIVO])
joblib.dump(pipeline, "modelo_pm25.joblib")                 # el ARTEFACTO que se despliega
print("Entrenado con", len(entrenamiento), "días-estación (2020–2024) → modelo_pm25.joblib")

# ── Desplegar empieza por CARGAR el artefacto (como lo haría otro equipo u otro servidor) ──
modelo = joblib.load("modelo_pm25.joblib")
ESTACIONES = sorted(entrenamiento["estacion"].unique())     # las que el modelo conoce
mae_prueba = mean_absolute_error(prueba[OBJETIVO], modelo.predict(prueba[FEATURES]))
mae_ingenuo_prueba = mean_absolute_error(prueba[OBJETIVO], prueba["pm25_hoy"])
print(f"MAE en prueba 2025: {mae_prueba:.2f} µg/m³  ·  línea base 'mañana = hoy': {mae_ingenuo_prueba:.2f}")
print("Filas de producción (2026):", len(produccion))

## 1. La función de predicción: el corazón del despliegue

Casi todo despliegue se reduce a **una función** que recibe datos nuevos y devuelve una predicción. Como guardamos el **pipeline completo**, la función no codifica la estación a mano: el pipeline ya sabe hacerlo.

### ✍️ Ejercicio 1 — Escribe `predecir_pm25` (predicción en línea)

In [ ]:
def predecir_pm25(estacion, pm25_hoy, pm25_ayer, media_7d, mes):
    entrada = pd.DataFrame({"estacion": [estacion], "pm25_hoy": [pm25_hoy], "pm25_ayer": [pm25_ayer],
                            "media_7d": [media_7d], "mes": [mes]})
    # TODO: usa modelo.predict(entrada), toma el elemento [0] y devuélvelo como float redondeado a 1 decimal
    return ...

print("PM2.5 estimado para mañana en Cerro Navia:", predecir_pm25("Cerro Navia", 60, 55, 48, 6), "µg/m³")

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    _e = pd.DataFrame({"estacion": ["Cerro Navia"], "pm25_hoy": [60], "pm25_ayer": [55], "media_7d": [48], "mes": [6]})
    esperado = round(float(modelo.predict(_e)[0]), 1)
    obtenido = predecir_pm25("Cerro Navia", 60, 55, 48, 6)
    assert isinstance(obtenido, float), "predecir_pm25 debe devolver un número (float)"
    assert abs(obtenido - esperado) < 0.05, f"se esperaba ~{esperado}"
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'return round(float(modelo.predict(entrada)[0]), 1)')
    print("   Detalle:", e)

## 2. Un despliegue robusto valida la entrada

Al modelo desplegado le van a llegar datos **malos**: una estación que no existe, un PM2.5 negativo, un **dato faltante** porque el monitor estuvo en mantención. Ojo: el bosque aleatorio de scikit-learn (≥1.4) **acepta NaN sin reclamar** y entrega un número igual — o sea, sin validación el modelo **inventa en silencio**. La validación es tuya.

Devolvemos un **diccionario** (así conversan las APIs web): `{"ok": True, "pm25_manana": ...}` o `{"ok": False, "error": "..."}`.

### ✍️ Ejercicio 2 — Escribe `predecir_seguro`

In [ ]:
def predecir_seguro(estacion, pm25_hoy, pm25_ayer, media_7d, mes):
    if estacion not in ESTACIONES:
        return {"ok": False, "error": f"Estación desconocida: {estacion}"}
    valores = [pm25_hoy, pm25_ayer, media_7d]
    # TODO 1: si algún valor es faltante (usa pd.isna) → {"ok": False, "error": "Dato faltante"}
    # TODO 2: si algún valor está fuera de 0–1000 µg/m³ → {"ok": False, "error": "Fuera de rango"}
    # TODO 3: si mes no está entre 1 y 12 → {"ok": False, "error": "Mes inválido"}
    return {"ok": True, "pm25_manana": predecir_pm25(estacion, pm25_hoy, pm25_ayer, media_7d, mes)}

print(predecir_seguro("Cerro Navia", 60, 55, 48, 6))
print(predecir_seguro("Cerro Navia", np.nan, 55, 48, 6))
print(predecir_seguro("Estación Inventada", 60, 55, 48, 6))

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    bueno = predecir_seguro("Cerro Navia", 60, 55, 48, 6)
    assert bueno["ok"] and abs(bueno["pm25_manana"] - predecir_pm25("Cerro Navia", 60, 55, 48, 6)) < 0.05, "el caso bueno debe predecir"
    assert not predecir_seguro("Cerro Navia", np.nan, 55, 48, 6)["ok"], "un dato faltante debe rechazarse"
    assert not predecir_seguro("Cerro Navia", -5, 55, 48, 6)["ok"], "un PM2.5 negativo debe rechazarse"
    assert not predecir_seguro("Cerro Navia", 60, 55, 48, 13)["ok"], "el mes 13 debe rechazarse"
    assert not predecir_seguro("Estación Inventada", 60, 55, 48, 6)["ok"]
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Revisa con any(pd.isna(v) for v in valores), luego all(0 <= v <= 1000 for v in valores) y mes in range(1, 13).')
    print("   Detalle:", e)

## 3. Puntuación por lotes en producción (con faltantes reales)

Ahora la tabla entera: `produccion` tiene **todas las estaciones y todos los días de 2026**. Las filas con algún feature faltante **no se predicen** (no inventamos): quedan con `pm25_predicho` vacío y `estado = "sin_prediccion"`, para que se vean y se cuenten.

### ✍️ Ejercicio 3 — Escribe `puntuar_lote` y aplícalo a 2026

In [ ]:
def puntuar_lote(tabla):
    resultado = tabla.copy()
    completo = resultado[FEATURES].notna().all(axis=1)     # True si la fila trae todos los datos
    resultado["pm25_predicho"] = np.nan
    # TODO: predice SOLO las filas completas, redondeado a 1 decimal:
    #       resultado.loc[completo, "pm25_predicho"] = modelo.predict(...).round(1)
    resultado["estado"] = np.where(completo, "ok", "sin_prediccion")
    return resultado

pred_2026 = puntuar_lote(produccion)
# TODO: cuántas filas quedaron con estado "sin_prediccion"
n_sin_prediccion = ...
print("filas:", len(pred_2026), "| sin predicción:", n_sin_prediccion)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    compl = produccion[FEATURES].notna().all(axis=1).values
    assert len(pred_2026) == len(produccion), "debe haber una fila por cada fila de producción"
    assert pred_2026["pm25_predicho"].notna().values.tolist() == compl.tolist(), "predice solo (y todas) las filas completas"
    esp = modelo.predict(produccion.loc[compl, FEATURES]).round(1)
    assert np.allclose(pred_2026.loc[compl, "pm25_predicho"].values, esp), "las predicciones no calzan con el modelo"
    assert n_sin_prediccion == int((~compl).sum())
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'resultado.loc[completo, "pm25_predicho"] = modelo.predict(resultado.loc[completo, FEATURES]).round(1); n_sin_prediccion = int((pred_2026["estado"] == "sin_prediccion").sum()).')
    print("   Detalle:", e)

In [ ]:
# ¿Dónde se concentran las filas sin predicción? (proporción por estación)
try:
    print((pred_2026["estado"] == "sin_prediccion").groupby(pred_2026["estacion"]).mean()
          .sort_values(ascending=False).round(2).head(6))
except Exception as e:
    print("Completa el ejercicio 3 para ver este resumen.", e)

## 4. Entregar el resultado

El producto de un lote es un **archivo** que alguien más usa. Dos detalles de oficio: decir explícitamente **para qué día** es cada pronóstico (`fecha_objetivo` = fecha + 1 día) y dejar visible el `estado` de cada fila.

### ✍️ Ejercicio 4 — Genera `predicciones_pm25_2026.csv`

In [ ]:
entregable = pred_2026[["fecha", "estacion", "pm25_predicho", "estado"]].copy()
# TODO: agrega la columna "fecha_objetivo" = fecha + 1 día  (pista: pd.Timedelta(days=1))
# TODO: guarda entregable en "predicciones_pm25_2026.csv" sin el índice
...
print("archivo creado:", os.path.exists("predicciones_pm25_2026.csv"))

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert "fecha_objetivo" in entregable.columns, "falta la columna fecha_objetivo"
    assert (entregable["fecha_objetivo"] - entregable["fecha"]).dt.days.eq(1).all()
    leido = pd.read_csv("predicciones_pm25_2026.csv")
    assert {"fecha_objetivo", "pm25_predicho", "estado"} <= set(leido.columns)
    assert len(leido) == len(entregable)
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'entregable["fecha_objetivo"] = entregable["fecha"] + pd.Timedelta(days=1); luego entregable.to_csv("predicciones_pm25_2026.csv", index=False).')
    print("   Detalle:", e)

## 5. Monitoreo de desempeño: ¿sigue funcionando?

En producción la verdad llega **un día después**: mañana sabremos cuánto marcó la estación. Con eso calculamos el **MAE en 2026**. Pero ojo con la comparación: la prueba 2025 cubre **todo el año**, y 2026 solo **enero–septiembre**, que incluye el invierno (cuando el PM2.5 es más alto y más difícil). La comparación **justa** es contra la prueba de los **mismos meses**. Y siempre contra la línea base: ¿el modelo le sigue ganando a «mañana = hoy»?

### ✍️ Ejercicio 5 — MAE en 2026 vs prueba (mismos meses) vs línea base

In [ ]:
con_verdad = pred_2026.dropna(subset=["pm25_predicho", "pm25_manana"])   # días con predicción Y dato real
# TODO: MAE en producción 2026 (pm25_manana real vs pm25_predicho)
mae_2026 = ...
# TODO: filas de `prueba` con mes <= 9, y su MAE (usa modelo.predict sobre sus FEATURES)
prueba_mismos_meses = ...
mae_prueba_mismos_meses = ...
# TODO: MAE de la línea base "mañana = hoy" en 2026 (pm25_manana vs pm25_hoy de con_verdad)
mae_ingenuo_2026 = ...
print("MAE 2026:", mae_2026, "| prueba ene–sep:", mae_prueba_mismos_meses, "| ingenuo 2026:", mae_ingenuo_2026)

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    _cv = pred_2026.dropna(subset=["pm25_predicho", "pm25_manana"])
    assert len(_cv) > 0, "primero completa el ejercicio 3"
    assert abs(mae_2026 - mean_absolute_error(_cv["pm25_manana"], _cv["pm25_predicho"])) < 1e-6
    _pm = prueba[prueba["fecha"].dt.month <= 9]
    assert abs(mae_prueba_mismos_meses - mean_absolute_error(_pm[OBJETIVO], modelo.predict(_pm[FEATURES]))) < 1e-6
    assert abs(mae_ingenuo_2026 - mean_absolute_error(_cv["pm25_manana"], _cv["pm25_hoy"])) < 1e-6
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'mean_absolute_error(con_verdad["pm25_manana"], con_verdad["pm25_predicho"]); prueba[prueba["fecha"].dt.month <= 9].')
    print("   Detalle:", e)

## 6. *Drift*: ¿los datos de entrada cambiaron?

El desempeño se mide con días de atraso; los **datos de entrada** se pueden vigilar el mismo día. Un cambio en la distribución de la entrada se llama ***drift***. El más básico y el más olvidado: la **proporción de faltantes**. Si una estación deja de reportar, el modelo no puede predecir para ella, aunque «el sistema funcione».

### ✍️ Ejercicio 6 — Compara faltantes (y niveles) entre historia y 2026

In [ ]:
falt_hist = historia["pm25"].isna().groupby(historia["estacion"]).mean()
# TODO: la misma proporción de faltantes, por estación, en los datos nuevos (2026)
falt_2026 = ...
# TODO: lista ORDENADA (sorted) de estaciones cuyo faltantes_2026 supera 0.20
estaciones_alerta = ...
print("faltantes historia:", falt_hist.round(3).to_dict())
print("Estaciones en alerta:", estaciones_alerta)

In [ ]:
# ── Celda de chequeo · Ejercicio 6 ──────────────────────────────────────────
try:
    _esp = nuevos["pm25"].isna().groupby(nuevos["estacion"]).mean()
    assert np.allclose(pd.Series(falt_2026).reindex(_esp.index).astype(float).values, _esp.values, atol=1e-3), "falt_2026 no calza: proporción de pm25 nulo por estación en `nuevos`"
    assert list(estaciones_alerta) == sorted(_esp[_esp > 0.20].index), "revisa el umbral (> 0.20) y ordena con sorted"
    print("✅ Ejercicio 6: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'nuevos["pm25"].isna().groupby(nuevos["estacion"]).mean(); luego sorted(falt_2026[falt_2026 > 0.20].index).')
    print("   Detalle:", e)

## 7. La ficha del modelo (*model card*)

Un modelo desplegado sin documentación es una caja negra con consecuencias. La **ficha de modelo** dice qué hace, con qué datos, cuánto se equivoca (con números **verificados**), **para qué NO usarlo** y quién responde. Honestidad ante todo: si apenas le gana a la línea base, se dice.

### ✍️ Ejercicio 7 — Completa la ficha con los números reales

In [ ]:
ficha = {
    "nombre": "Pronóstico de PM2.5 del día siguiente por estación",
    "version": "1.0 — entrenado con 2020–2024, probado con 2025",
    "uso_previsto": ...,      # TODO: para qué y para quién (con revisión humana)
    "no_usar_para": ...,      # TODO: qué decisiones NO debe tomar este modelo
    "datos": "SINCA: PM2.5 diario de 14 estaciones (2020–2025). Producción: ene–sep 2026.",
    "mae_prueba": ...,        # TODO: round(mae_prueba, 2)
    "mae_linea_base": round(mae_ingenuo_prueba, 2),
    "mae_2026": ...,          # TODO: round(mae_2026, 2)
    "limitaciones": ...,      # TODO: qué no sabe hacer, con honestidad
    "monitoreo": "MAE mensual vs prueba de los mismos meses; faltantes por estación.",
    "responsable": "Equipo de datos (completa con el nombre real)",
}
from IPython.display import Markdown, display
display(Markdown("\n".join(f"- **{k}:** {v}" for k, v in ficha.items())))

In [ ]:
# ── Celda de chequeo · Ejercicio 7 ──────────────────────────────────────────
try:
    _claves = {"nombre", "uso_previsto", "no_usar_para", "datos", "mae_prueba", "mae_linea_base",
               "mae_2026", "limitaciones", "monitoreo", "responsable"}
    assert _claves <= set(ficha), f"faltan claves: {_claves - set(ficha)}"
    for _k in ["uso_previsto", "no_usar_para", "limitaciones"]:
        assert isinstance(ficha[_k], str) and len(ficha[_k]) > 15, f"escribe '{_k}' con una frase real"
    assert abs(ficha["mae_prueba"] - mae_prueba) < 0.01
    assert abs(ficha["mae_2026"] - mae_2026) < 0.01
    print("✅ Ejercicio 7: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Reemplaza cada ... por una frase (texto entre comillas) o por round(mae_prueba, 2) y round(mae_2026, 2).')
    print("   Detalle:", e)

## 8. (Opcional) Una interfaz para personas: Gradio

`predecir_seguro` es **exactamente** lo que envuelven las formas más avanzadas de despliegue: una
API web (FastAPI/Flask) o una app sencilla con **Gradio**, como la de R1-09. En Colab, instala
con `!pip -q install gradio` y descomenta `demo.launch(share=True)`. La celda está protegida para
no bloquear el cuaderno si Gradio no está instalado.

In [ ]:
# ── App Gradio (opcional; en Colab) ──────────────────────────────────────────
try:
    import gradio as gr

    def app(estacion, pm25_hoy, pm25_ayer, media_7d, mes):
        r = predecir_seguro(estacion, pm25_hoy, pm25_ayer, media_7d, int(mes))
        if not r["ok"]:
            return "⚠️ " + r["error"]
        aviso = " — sobre la norma diaria (50 µg/m³)" if r["pm25_manana"] > 50 else ""
        return f"PM2.5 estimado para mañana: {r['pm25_manana']} µg/m³{aviso}"

    demo = gr.Interface(
        app,
        [gr.Dropdown(ESTACIONES, value="Cerro Navia", label="Estación"),
         gr.Number(value=40, label="PM2.5 hoy (µg/m³)"),
         gr.Number(value=35, label="PM2.5 ayer (µg/m³)"),
         gr.Number(value=30, label="Media últimos 7 días (µg/m³)"),
         gr.Slider(1, 12, value=6, step=1, label="Mes")],
        gr.Text(label="Pronóstico"),
        title="Pronóstico de PM2.5 del día siguiente (uso interno, con revisión humana)",
    )
    # demo.launch(share=True)   # ← descomenta en Colab para obtener un link
    print("✔ App Gradio construida. En Colab ejecuta: demo.launch(share=True)")
except ModuleNotFoundError:
    print("ℹ Gradio no está instalado aquí. En Colab corre primero:  !pip -q install gradio")
except Exception as e:
    print("ℹ La app no se construyó (¿completaste los ejercicios 1 y 2?):", e)

## Cierre

- Desplegar = **cargar el artefacto** + una **función de predicción** + **validar la entrada** +
  **puntuar lotes** + **entregar un archivo** que otro pueda usar.
- En producción **no se inventa**: las filas con datos faltantes se marcan y se cuentan
  (en 2026, una de cada ~7 filas no se pudo predecir, concentradas en 4 estaciones).
- **Monitorear** es comparar con justicia: la misma época del año y contra la línea base. El
  MAE de 2026 es mayor que el de la prueba 2025 completa, pero buena parte se explica porque
  enero–septiembre incluye el invierno.
- El ***drift* de faltantes** se vigila el mismo día; el desempeño, con días de atraso.
- La **ficha de modelo** acompaña al modelo siempre: qué hace, cuánto se equivoca, para qué NO
  usarlo y quién responde.

> *Ética distribuida:* un pronóstico de calidad del aire puede mover decisiones que afectan la
> salud de la gente. Por eso «con revisión humana» no es una frase de relleno en la ficha: es el
> diseño. *En la profundización verás **dónde** se degrada el modelo (y por qué) y cómo elegir
> un **umbral de alerta**.*